# DSA 4060 – Practical Quiz: Personalized Movie Recommender

| Field | Detail |
|---|---|
| **Name** | Hans Onesmo |
| **Student ID** | 671463 |
| **Assigned User ID** | 24 |

**Assigned User ID calculation:** last two digits = 63 → 63 MOD 40 = 23 → 23 + 1 = **24**

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

STUDENT_ID = 671463
ASSIGNED_USER_ID = (int(str(STUDENT_ID)[-2:]) % 40) + 1
print("Last two digits:", str(STUDENT_ID)[-2:])
print("Assigned User ID:", ASSIGNED_USER_ID)

Last two digits: 63
Assigned User ID: 24


## Task 1 – Load and Inspect the Data

In [2]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
print("movies.csv – first five rows")
display(movies.head())

movies.csv – first five rows


,movie_id,title,genres
0,1,Inception,Sci-Fi|Thriller
1,2,Interstellar,Sci-Fi|Drama
2,3,The Matrix,Sci-Fi|Action
3,4,Arrival,Sci-Fi|Drama
4,5,Edge of Tomorrow,Sci-Fi|Action


In [3]:
print("ratings.csv – first five rows")
display(ratings.head())

ratings.csv – first five rows


,user_id,movie_id,rating
0,1,2,3.5
1,1,3,4.5
2,1,5,3.5
3,1,6,3.5
4,1,11,3.0


In [4]:
print(f"movies.csv : {movies.shape[0]} rows, {movies.shape[1]} columns")
print(f"ratings.csv: {ratings.shape[0]} rows, {ratings.shape[1]} columns")
print("\nUnique users in ratings:", ratings['user_id'].nunique())
print("Unique movies in catalogue:", movies['movie_id'].nunique())

movies.csv : 36 rows, 3 columns
ratings.csv: 440 rows, 3 columns

Unique users in ratings: 40
Unique movies in catalogue: 36


In [5]:
print("Missing values – movies.csv")
print(movies.isnull().sum())
print("\nMissing values – ratings.csv")
print(ratings.isnull().sum())
print("\nTotal missing: movies =", int(movies.isnull().sum().sum()),
      "| ratings =", int(ratings.isnull().sum().sum()))

Missing values – movies.csv
movie_id    0
title       0
genres      0
dtype: int64

Missing values – ratings.csv
user_id     0
movie_id    0
rating      0
dtype: int64

Total missing: movies = 0 | ratings = 0


**Missing-value finding:** No missing values were found in either dataset. `movies.csv` (36 rows) and `ratings.csv` (440 rows) have zero nulls in every column, so no cleaning or imputation was needed.

## Task 2 – Profile the Assigned User (User 24)

In [6]:
user_ratings = (
    ratings[ratings["user_id"] == ASSIGNED_USER_ID]
    .merge(movies, on="movie_id", how="left")
    [["movie_id", "title", "genres", "rating"]]
    .sort_values("rating", ascending=False)
    .reset_index(drop=True)
)
print(f"User {ASSIGNED_USER_ID} has rated {len(user_ratings)} movies.")
display(user_ratings)

User 24 has rated 13 movies.


,movie_id,title,genres,rating
0,17,The Pursuit of Happyness,Drama|Biography,4.5
1,19,The Social Network,Drama|Biography,4.5
2,2,Interstellar,Sci-Fi|Drama,4.0
3,6,Blade Runner 2049,Sci-Fi|Drama,3.5
4,20,Hidden Figures,Drama|Biography,3.5
5,1,Inception,Sci-Fi|Thriller,3.0
6,14,The Notebook,Romance|Drama,2.5
7,27,The Conjuring,Horror,2.5
8,7,The Dark Knight,Action|Crime,2.0
9,9,Mad Max: Fury Road,Action|Adventure,1.5


In [7]:
top3 = user_ratings.nlargest(3, "rating", keep="first")[["title", "genres", "rating"]].reset_index(drop=True)
print("Top three highest-rated movies:")
display(top3)

Top three highest-rated movies:


,title,genres,rating
0,The Pursuit of Happyness,Drama|Biography,4.5
1,The Social Network,Drama|Biography,4.5
2,Interstellar,Sci-Fi|Drama,4.0


In [8]:
# Evidence for genre preference: how often each genre appears, and the user's average rating per genre
sep = r"[|,;]"
g = user_ratings.assign(genre=user_ratings["genres"].str.split(sep, regex=True)).explode("genre")
g["genre"] = g["genre"].str.strip()
genre_summary = (
    g.groupby("genre")["rating"]
     .agg(movies_rated="count", avg_rating="mean")
     .round(2)
     .sort_values(["avg_rating", "movies_rated"], ascending=False)
)
liked_g = g[g["rating"] >= 4.0]["genre"].value_counts().rename("count_in_liked_movies (>=4.0)")
genre_summary = genre_summary.join(liked_g).fillna({"count_in_liked_movies (>=4.0)": 0}).astype({"count_in_liked_movies (>=4.0)": int})
display(genre_summary)

,movies_rated,avg_rating,count_in_liked_movies (>=4.0)
genre,,,
Biography,3,4.17,2
Drama,6,3.75,3
Sci-Fi,3,3.50,1
Thriller,1,3.00,0
Horror,1,2.50,0
Romance,2,2.00,0
Crime,1,2.00,0
Action,3,1.67,0
Adventure,2,1.50,0


**Preferred genre(s) – interpretation:** User 24 rated 13 movies, but only 3 are rated 4.0 or higher (The Pursuit of Happyness 4.5, The Social Network 4.5, Interstellar 4.0), and all three are Dramas. Drama is the clearest preference: it appears in 6 of the 13 rated movies with an average rating of 3.75, and in 3 of 3 liked movies. Biography has the highest average (4.17 across 3 movies, 2 of them rated 4.0+), suggesting the user especially enjoys true-story dramas. In contrast, Action, Adventure, Animation, Musical and Superhero movies all average 1.67 or lower, so the user clearly dislikes those genres.

## Task 3 – Build the Recommendation Logic (Content-Based, Genres)

**Step 1 – Movies the user likes (rating ≥ 4.0).**

In [9]:
liked = user_ratings[user_ratings["rating"] >= 4.0].copy()
print(f"Liked movies (rating >= 4.0): {len(liked)}")
display(liked[["title", "genres", "rating"]])

Liked movies (rating >= 4.0): 3


,title,genres,rating
0,The Pursuit of Happyness,Drama|Biography,4.5
1,The Social Network,Drama|Biography,4.5
2,Interstellar,Sci-Fi|Drama,4.0


**Step 2 – Convert genres to numerical features** using `CountVectorizer` (each genre becomes one binary feature; genres are split on `|`, `,` or `;` so multi-word genres like *Sci-Fi* stay intact).

In [10]:
import re
movies = movies.copy()
movies["genres"] = movies["genres"].fillna("")

def genre_tokenizer(text):
    return [t.strip().lower() for t in re.split(r"[|,;]", text) if t.strip()]

vectorizer = CountVectorizer(tokenizer=genre_tokenizer, token_pattern=None, binary=True)
genre_matrix = vectorizer.fit_transform(movies["genres"])

print("Genre feature matrix shape (movies x genres):", genre_matrix.shape)
print("Genre vocabulary:", list(vectorizer.get_feature_names_out()))

Genre feature matrix shape (movies x genres): (36, 16)
Genre vocabulary: ['action', 'adventure', 'animation', 'biography', 'comedy', 'crime', 'drama', 'family', 'horror', 'musical', 'mystery', 'romance', 'sci-fi', 'sports', 'superhero', 'thriller']


**Step 3 – Cosine similarity.** The user's taste profile is the *rating-weighted average* of the genre vectors of the movies they like. Every catalogue movie is then scored by its cosine similarity to that profile.

In [11]:
movie_index = {mid: i for i, mid in enumerate(movies["movie_id"])}
liked_idx = [movie_index[m] for m in liked["movie_id"] if m in movie_index]
weights = liked["rating"].values[: len(liked_idx)]

liked_vectors = genre_matrix[liked_idx].toarray()
user_profile = np.average(liked_vectors, axis=0, weights=weights).reshape(1, -1)

# Similarity of every movie to the user's taste profile
profile_sim = cosine_similarity(user_profile, genre_matrix.toarray()).ravel()

# Similarity of every movie to each individual liked movie (used for explanations)
sim_to_liked = cosine_similarity(genre_matrix, genre_matrix[liked_idx])   # (n_movies x n_liked)

print("User taste profile (non-zero genre weights):")
profile_series = pd.Series(user_profile.ravel(), index=vectorizer.get_feature_names_out())
display(profile_series[profile_series > 0].sort_values(ascending=False).round(3).to_frame("weight"))

User taste profile (non-zero genre weights):


,weight
drama,1.000
biography,0.692
sci-fi,0.308


**Step 4 – Exclude already-rated movies and return exactly five.** Ties in similarity are broken by the movie's average rating across all users, then by title, so the result is deterministic.

In [12]:
avg_rating_all = ratings.groupby("movie_id")["rating"].mean()
n_ratings_all = ratings.groupby("movie_id")["rating"].count()

scores = movies.copy()
scores["similarity"] = profile_sim
scores["community_avg"] = scores["movie_id"].map(avg_rating_all).fillna(0)
scores["n_ratings"] = scores["movie_id"].map(n_ratings_all).fillna(0).astype(int)

already_rated = set(user_ratings["movie_id"])
candidates = scores[~scores["movie_id"].isin(already_rated)]

top5 = (
    candidates
    .sort_values(["similarity", "community_avg", "title"], ascending=[False, False, True])
    .head(5)
    .reset_index(drop=True)
)

# Safety checks required by the quiz
assert len(top5) == 5, "Must return exactly five recommendations"
assert not set(top5["movie_id"]) & already_rated, "A recommended movie was already rated by the user!"
print("Checks passed: exactly 5 recommendations, none already rated by User", ASSIGNED_USER_ID)

display(top5[["title", "genres", "similarity", "community_avg", "n_ratings"]].round(3))

Checks passed: exactly 5 recommendations, none already rated by User 24


,title,genres,similarity,community_avg,n_ratings
0,The Shawshank Redemption,Drama,0.797,3.333,12
1,Arrival,Sci-Fi|Drama,0.737,3.147,17
2,Remember the Titans,Sports|Drama,0.564,3.550,10
3,Ford v Ferrari,Sports|Drama,0.564,3.529,17
4,Moneyball,Sports|Drama,0.564,3.344,16


## Task 4 – Explain the Recommendations

In [13]:
liked_titles = liked["title"].tolist()
liked_genres = liked["genres"].tolist()
user_top_genres = profile_series.sort_values(ascending=False)
user_top_genres = [g for g in user_top_genres.index if user_top_genres[g] > 0]

rows = []
for rank, row in top5.iterrows():
    i = movie_index[row["movie_id"]]
    j = int(np.argmax(sim_to_liked[i]))                      # most similar liked movie
    anchor_title, anchor_genres = liked_titles[j], liked_genres[j]
    rec_genres = set(genre_tokenizer(row["genres"]))
    shared_with_anchor = sorted(rec_genres & set(genre_tokenizer(anchor_genres)))
    shared_with_profile = [g for g in user_top_genres if g in rec_genres][:3]

    reason = (
        f"Most similar to liked movie '{anchor_title}' "
        f"(cosine = {sim_to_liked[i, j]:.2f}; shared genres: {', '.join(shared_with_anchor) or 'none'}). "
        f"Matches the user's strongest genres: {', '.join(shared_with_profile) or 'n/a'}. "
        f"Overall similarity to taste profile = {row['similarity']:.2f}."
    )
    rows.append({
        "Rank": rank + 1,
        "Recommended Movie": row["title"],
        "Genres": row["genres"],
        "Reason": reason,
    })

results = pd.DataFrame(rows)
display(results.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

Rank,Recommended Movie,Genres,Reason
1,The Shawshank Redemption,Drama,Most similar to liked movie 'The Pursuit of Happyness' (cosine = 0.71; shared genres: drama). Matches the user's strongest genres: drama. Overall similarity to taste profile = 0.80.
2,Arrival,Sci-Fi|Drama,"Most similar to liked movie 'Interstellar' (cosine = 1.00; shared genres: drama, sci-fi). Matches the user's strongest genres: drama, sci-fi. Overall similarity to taste profile = 0.74."
3,Remember the Titans,Sports|Drama,Most similar to liked movie 'The Pursuit of Happyness' (cosine = 0.50; shared genres: drama). Matches the user's strongest genres: drama. Overall similarity to taste profile = 0.56.
4,Ford v Ferrari,Sports|Drama,Most similar to liked movie 'The Pursuit of Happyness' (cosine = 0.50; shared genres: drama). Matches the user's strongest genres: drama. Overall similarity to taste profile = 0.56.
5,Moneyball,Sports|Drama,Most similar to liked movie 'The Pursuit of Happyness' (cosine = 0.50; shared genres: drama). Matches the user's strongest genres: drama. Overall similarity to taste profile = 0.56.


## Task 5 – Critical Thinking

**Limitation:** The recommender uses only genre labels, so movies with the same genres look identical to it. For example, Remember the Titans, Ford v Ferrari and Moneyball (all Sports|Drama) tied at a similarity of 0.564, and the tie could only be broken with the community average rating, not with anything specific to User 24. The profile is also built from just three liked movies, which makes it thin and pushes the list towards Drama only.

**Improvement:** Make it a hybrid recommender by adding collaborative filtering on top of the genre scores. Using the ratings of the other 39 users, I could find users whose ratings are most similar to User 24 and boost movies those users rated highly. This would separate movies with identical genres and bring in taste signals that genres alone cannot capture.